# FairyNews — финальный ноутбук (этап 4)

- Рабочая папка терминала Jupyter: **корень репозитория** (рядом с каталогом `app`).
- Установка: `pip install -r requirements.txt`.
- Снимок RAG: `data/notebook_rag_snapshot.json`. Полный индекс: вручную `python -m rag --reset`.
- Без `OPENAI_API_KEY` используется **stub** LLM; с ключом — OpenAI (или совместимый `OPENAI_BASE_URL`).

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

# По умолчанию весь проект — родитель каталога notebooks/
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / "app").is_dir() else NOTEBOOK_DIR.parent
SNAPSHOT_DEFAULT = REPO_ROOT / "data" / "notebook_rag_snapshot.json"
REPORTS_DEFAULT = REPO_ROOT / "data" / "reports" / "runs"


def ensure_repo_root(repo_root: Path = REPO_ROOT) -> Path:
    if not (repo_root / "app").is_dir():
        raise RuntimeError(
            "Укажите корень репозитория: рядом должна быть папка app/"
        )
    return repo_root.resolve()


def configure_environment(
    *,
    use_snapshot_rag: bool = True,
    use_stub_llm: bool = False,
    reports_dir: Path | None = None,
    repo_root: Path = REPO_ROOT,
) -> Path:
    """Задаёт переменные окружения для одного прогона (без глобального состояния снаружи)."""
    root = ensure_repo_root(repo_root)
    os.environ["FAIRYNEWS_RAG_BACKEND"] = "snapshot" if use_snapshot_rag else "chroma"
    if use_stub_llm or not os.environ.get("OPENAI_API_KEY"):
        os.environ["FAIRYNEWS_LLM_MODE"] = "stub"
    else:
        os.environ.pop("FAIRYNEWS_LLM_MODE", None)
    rd = reports_dir if reports_dir is not None else REPORTS_DEFAULT
    os.environ["FAIRYNEWS_REPORTS_DIR"] = str(rd)
    os.environ["FAIRYNEWS_SAVE_REPORTS"] = "1"
    return root

In [ ]:
import sys
from pathlib import Path
from typing import Any

ensure_repo_root(REPO_ROOT)
_root_s = str(REPO_ROOT)
if _root_s not in sys.path:
    sys.path.insert(0, _root_s)

from app.agents_pipeline import default_snapshot_path, run_four_agent_pipeline
from app.presets import get_preset


def run_story_pipeline(
    news_text: str,
    preset_id: str = "default",
    *,
    snapshot_path: Path | None = None,
    use_snapshot_rag: bool = True,
    use_stub_llm: bool = False,
    reports_dir: Path | None = None,
    repo_root: Path = REPO_ROOT,
) -> dict[str, Any]:
    """Оркестратор: новость → 4 агента → результат и блок ``report``."""
    configure_environment(
        use_snapshot_rag=use_snapshot_rag,
        use_stub_llm=use_stub_llm,
        reports_dir=reports_dir,
        repo_root=repo_root,
    )
    preset = get_preset(preset_id)
    domains = preset.get("domains")
    snap = snapshot_path or default_snapshot_path()
    return run_four_agent_pipeline(
        news_text,
        str(preset["retrieval_hint"]),
        domains,
        preset_id=preset_id,
        news_id=None,
        snapshot_path=snap,
        rag_backend="snapshot" if use_snapshot_rag else "chroma",
    )


def save_report_if_needed(
    pipeline_out: dict[str, Any],
    *,
    repo_root: Path = REPO_ROOT,
) -> str | None:
    """Сохраняет JSON отчёт; путь к каталогу — из окружения или по умолчанию."""
    from app.report_storage import save_run_report

    report = pipeline_out.get("report")
    if not report:
        return None
    ensure_repo_root(repo_root)
    return save_run_report(report)

In [ ]:
def run_all_demo(
    news_text: str,
    preset_id: str = "default",
    *,
    snapshot_path: Path | None = None,
    use_snapshot_rag: bool = True,
    use_stub_llm: bool = False,
) -> dict[str, Any]:
    """Главная функция для проверки: пайплайн + сохранение отчёта."""
    out = run_story_pipeline(
        news_text,
        preset_id,
        snapshot_path=snapshot_path,
        use_snapshot_rag=use_snapshot_rag,
        use_stub_llm=use_stub_llm,
        reports_dir=REPORTS_DEFAULT,
        repo_root=REPO_ROOT,
    )
    rid = save_report_if_needed(out, repo_root=REPO_ROOT)
    return {
        "tale": out["tale"],
        "chosen_tale_source": out["chosen_tale_source"],
        "news_brief": out["news_brief"],
        "qa": out["qa"],
        "run_id": rid,
        "report_path": str(REPORTS_DEFAULT / f"{rid}.json") if rid else None,
    }

In [ ]:
# Тестовая ячейка: введите новость и пресет
news = input("Текст новости (или выжимка): ").strip()
preset = input("preset_id [default]: ").strip() or "default"
stub = input("Принудительно stub LLM? y/N: ").strip().lower() in ("y", "д")

result = run_all_demo(
    news,
    preset,
    use_snapshot_rag=True,
    use_stub_llm=stub,
)
print("run_id:", result.get("run_id"))
print("отчёт:", result.get("report_path"))
print("\n--- Сказка ---\n")
print(result["tale"])